In [2]:
"""
Pipeline de Data Scraping pour Tenymalagasy.org
Pipeline complet pour extraire et traiter les données linguistiques malgaches
"""

import requests
from bs4 import BeautifulSoup
import json
import csv
import sqlite3
import time
from datetime import datetime
from typing import Dict, List, Optional
import re
from pathlib import Path
import logging

# Configuration du logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('scraping.log'),
        logging.StreamHandler()
    ]
)

class MalagasyDataPipeline:
    """Pipeline complet pour le scraping et traitement de données malgaches"""
    
    def __init__(self, base_url: str = "https://tenymalagasy.org", 
                 db_path: str = "malagasy_data.db",
                 delay: float = 2.0):
        self.base_url = base_url
        self.db_path = db_path
        self.delay = delay  # Délai entre les requêtes (respect du serveur)
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'})
        self.init_database()
    
    def init_database(self):
        """Initialise la base de données SQLite"""
        conn = sqlite3.connect(self.db_path)
        c = conn.cursor()
        
        # Table principale pour les mots
        c.execute('''
            CREATE TABLE IF NOT EXISTS words (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                word TEXT UNIQUE NOT NULL,
                definition TEXT,
                etymology TEXT,
                examples TEXT,
                category TEXT,
                url TEXT,
                scraped_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
        ''')
        
        # Table pour les traductions
        c.execute('''
            CREATE TABLE IF NOT EXISTS translations (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                word_id INTEGER,
                language TEXT,
                translation TEXT,
                FOREIGN KEY (word_id) REFERENCES words (id)
            )
        ''')
        
        # Table pour les statistiques
        c.execute('''
            CREATE TABLE IF NOT EXISTS scraping_stats (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                date TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                words_scraped INTEGER,
                errors INTEGER,
                duration_seconds REAL
            )
        ''')
        
        conn.commit()
        conn.close()
        logging.info("Base de données initialisée")
    
    def fetch_page(self, url: str) -> Optional[str]:
        """Récupère le contenu d'une page avec gestion d'erreurs"""
        try:
            time.sleep(self.delay)  # Respect du serveur
            response = self.session.get(url, verify=False)
            response.raise_for_status()
            return response.text
        except requests.RequestException as e:
            logging.error(f"Erreur lors de la récupération de {url}: {e}")
            return None
    
    def parse_word_page(self, html: str, url: str) -> Optional[Dict]:
        """Parse une page de mot et extrait les données"""
        try:
            soup = BeautifulSoup(html, 'html.parser')
            
            data = {
                'word': '',
                'definition': '',
                'etymology': '',
                'examples': [],
                'category': '',
                'translations': {},
                'url': url
            }
            
            # Extraction du mot principal (à adapter selon la structure réelle)
            title = soup.find('h1') or soup.find('title')
            if title:
                data['word'] = title.get_text(strip=True)
            
            # Extraction de la définition
            definition_elem = soup.find('div', class_='definition') or soup.find('p')
            if definition_elem:
                data['definition'] = definition_elem.get_text(strip=True)
            
            # Extraction des exemples
            examples = soup.find_all('div', class_='example') or soup.find_all('li')
            data['examples'] = [ex.get_text(strip=True) for ex in examples[:5]]
            
            # Extraction de l'étymologie
            etym = soup.find('div', class_='etymology')
            if etym:
                data['etymology'] = etym.get_text(strip=True)
            
            return data
        except Exception as e:
            logging.error(f"Erreur lors du parsing: {e}")
            return None
    
    def scrape_word(self, word: str) -> Optional[Dict]:
        """Scrape les données d'un mot spécifique"""
        url = f"{self.base_url}/bins/teny2/{word}"
        logging.info(f"Scraping du mot: {word}")
        
        html = self.fetch_page(url)
        if not html:
            return None
        
        return self.parse_word_page(html, url)
    
    def scrape_word_list(self, words: List[str]) -> List[Dict]:
        """Scrape une liste de mots"""
        results = []
        start_time = time.time()
        errors = 0
        
        for i, word in enumerate(words, 1):
            logging.info(f"Progression: {i}/{len(words)}")
            data = self.scrape_word(word)
            
            if data:
                results.append(data)
                self.save_to_database(data)
            else:
                errors += 1
            
            # Pause tous les 10 mots
            if i % 10 == 0:
                time.sleep(5)
        
        duration = time.time() - start_time
        self.save_stats(len(results), errors, duration)
        
        return results
    
    def save_to_database(self, data: Dict):
        """Sauvegarde les données dans la base de données"""
        conn = sqlite3.connect(self.db_path)
        c = conn.cursor()
        
        try:
            c.execute('''
                INSERT OR REPLACE INTO words 
                (word, definition, etymology, examples, category, url)
                VALUES (?, ?, ?, ?, ?, ?)
            ''', (
                data['word'],
                data['definition'],
                data['etymology'],
                '\n'.join(data['examples']),
                data['category'],
                data['url']
            ))
            
            word_id = c.lastrowid
            
            # Sauvegarder les traductions
            for lang, trans in data.get('translations', {}).items():
                c.execute('''
                    INSERT INTO translations (word_id, language, translation)
                    VALUES (?, ?, ?)
                ''', (word_id, lang, trans))
            
            conn.commit()
            logging.info(f"Données sauvegardées pour: {data['word']}")
        except sqlite3.Error as e:
            logging.error(f"Erreur SQL: {e}")
        finally:
            conn.close()
    
    def save_stats(self, words_scraped: int, errors: int, duration: float):
        """Sauvegarde les statistiques de scraping"""
        conn = sqlite3.connect(self.db_path)
        c = conn.cursor()
        c.execute('''
            INSERT INTO scraping_stats (words_scraped, errors, duration_seconds)
            VALUES (?, ?, ?)
        ''', (words_scraped, errors, duration))
        conn.commit()
        conn.close()
    
    def export_to_json(self, output_file: str = "malagasy_data.json"):
        """Exporte les données en JSON"""
        conn = sqlite3.connect(self.db_path)
        c = conn.cursor()
        c.execute('SELECT * FROM words')
        
        columns = [desc[0] for desc in c.description]
        results = []
        
        for row in c.fetchall():
            results.append(dict(zip(columns, row)))
        
        conn.close()
        
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(results, f, ensure_ascii=False, indent=2)
        
        logging.info(f"Données exportées vers {output_file}")
        return results
    
    def export_to_csv(self, output_file: str = "malagasy_data.csv"):
        """Exporte les données en CSV"""
        conn = sqlite3.connect(self.db_path)
        c = conn.cursor()
        c.execute('SELECT word, definition, etymology, examples, url FROM words')
        
        with open(output_file, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['word', 'definition', 'etymology', 'examples', 'url'])
            writer.writerows(c.fetchall())
        
        conn.close()
        logging.info(f"Données exportées vers {output_file}")
    
    def get_statistics(self) -> Dict:
        """Récupère les statistiques de la base de données"""
        conn = sqlite3.connect(self.db_path)
        c = conn.cursor()
        
        c.execute('SELECT COUNT(*) FROM words')
        total_words = c.fetchone()[0]
        
        c.execute('SELECT AVG(duration_seconds), SUM(words_scraped) FROM scraping_stats')
        avg_duration, total_scraped = c.fetchone()
        
        conn.close()
        
        return {
            'total_words': total_words,
            'total_scraped': total_scraped or 0,
            'avg_duration': avg_duration or 0
        }
    
    def search_words(self, pattern: str) -> List[Dict]:
        """Recherche des mots dans la base de données"""
        conn = sqlite3.connect(self.db_path)
        c = conn.cursor()
        
        c.execute('''
            SELECT word, definition, url 
            FROM words 
            WHERE word LIKE ? OR definition LIKE ?
        ''', (f'%{pattern}%', f'%{pattern}%'))
        
        results = [{'word': r[0], 'definition': r[1], 'url': r[2]} 
                   for r in c.fetchall()]
        conn.close()
        
        return results


# Exemple d'utilisation
if __name__ == "__main__":
    # Initialiser le pipeline
    pipeline = MalagasyDataPipeline(delay=2.0)
    
    # Liste de mots d'exemple à scraper
    sample_words = [
        "teny", "malagasy", "fanazavana", "boky", 
        "firenena", "tanàna", "olona", "zavatra"
    ]
    
    print("=== Pipeline de Scraping Tenymalagasy.org ===\n")
    
    # Option 1: Scraper une liste de mots
    print("1. Scraping des mots d'exemple...")
    results = pipeline.scrape_word_list(sample_words)
    print(f"✓ {len(results)} mots scrapés avec succès\n")
    
    # Option 2: Exporter les données
    print("2. Export des données...")
    pipeline.export_to_json("malagasy_export.json")
    pipeline.export_to_csv("malagasy_export.csv")
    print("✓ Données exportées\n")
    
    # Option 3: Afficher les statistiques
    print("3. Statistiques:")
    stats = pipeline.get_statistics()
    for key, value in stats.items():
        print(f"   {key}: {value}")
    
    # Option 4: Rechercher des mots
    print("\n4. Recherche d'exemple:")
    search_results = pipeline.search_words("teny")
    for result in search_results[:3]:
        print(f"   - {result['word']}: {result['definition'][:50]}...")


2025-12-18 08:35:00,950 - INFO - Base de données initialisée
2025-12-18 08:35:00,953 - INFO - Progression: 1/8
2025-12-18 08:35:00,954 - INFO - Scraping du mot: teny


=== Pipeline de Scraping Tenymalagasy.org ===

1. Scraping des mots d'exemple...


C:\ProgramData\Anaconda3\lib\site-packages\urllib3\connectionpool.py:1045: InsecureRequestWarning: Unverified HTTPS request is being made to host 'tenymalagasy.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
2025-12-18 08:35:04,394 - INFO - Données sauvegardées pour: Rakibolana sy Rakipahalalana malagasy : teny
2025-12-18 08:35:04,395 - INFO - Progression: 2/8
2025-12-18 08:35:04,396 - INFO - Scraping du mot: malagasy
C:\ProgramData\Anaconda3\lib\site-packages\urllib3\connectionpool.py:1045: InsecureRequestWarning: Unverified HTTPS request is being made to host 'tenymalagasy.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
2025-12-18 08:35:06,815 - INFO - Données sauvegardées pour: Rakibolana sy Rakipahalalana malagasy : malagasy
2025-12-18 08:35:06,816 - INFO - Progression: 3

✓ 8 mots scrapés avec succès

2. Export des données...
✓ Données exportées

3. Statistiques:
   total_words: 8
   total_scraped: 8
   avg_duration: 22.6929190158844

4. Recherche d'exemple:
   - Rakibolana sy Rakipahalalana malagasy : teny: ...
